In [1]:
import random

In [2]:
# Función para sumar dos puntos en la curva elíptica
def sumaPuntos(P, Q, p):
    if P == (0, 1, 0): return Q
    if Q == (0, 1, 0): return P
    if Q[0]-P[0] == 0: return (0, 1, 0)
    num = (Q[1] - P[1]) % p
    den = (Q[0] - P[0]) % p
    try:
        inv_den = pow(den, -1, p)
    except ValueError:
        return (0, 1, 0)
    pendiente = (num * inv_den) % p
    x3 = (pendiente**2 - P[0] - Q[0]) % p
    y3 = (pendiente * (P[0] - x3) - P[1]) % p
    return (x3, y3, 1)

# Función para doblar un punto en la curva elíptica
def sum2P(P, a, p):
    if P == (0, 1, 0): return (0, 1, 0)
    try:
        inv = pow(2 * P[1], -1, p)
    except ValueError:
        return (0, 1, 0)
    pendiente = (3 * P[0]**2 + a) * inv % p
    x3 = (pendiente**2 - 2 * P[0]) % p
    y3 = (pendiente * (P[0] - x3) - P[1]) % p
    return (x3, y3, 1)

# Función para multiplicar un punto por un escalar
def right_left_kP(k, P, a, p):
    k_bin = bin(k)[2:] 
    Q = (0, 1, 0)
    for k_i in k_bin[::-1]:
        if k_i == '1':
            Q = sumaPuntos(Q, P, p)
        P = sum2P(P, a, p)
    return Q

In [3]:
def keysECDSA(p, a, b, G, q):
    d = random.randint(1, q - 1)
    B = right_left_kP(d, G, a, p)

    return (p, a, b, q, G, B),(d)

In [10]:
def firmaECDSA(m, kpriv, kpub):
    p, a, b, q, G, B = kpub
    d = kpriv
    
    k = random.randint(1, q - 1)
    T = right_left_kP(k, G, a, p)
    r = T[0] % q
    s = (m + r * d) * pow(k, -1, q) % q

    return (r, s)

In [5]:
def verificarECDSA(kpub, m, firma):
    r, s = firma
    p, a, b, q, G, B = kpub

    w = pow(s, -1, q)
    u1 = (m * w) % q
    u2 = (r * w) % q
    P = sumaPuntos(right_left_kP(u1, G, a, p), right_left_kP(u2, B, a, p), p)
    return P[0] % q == r

In [11]:
a , b = 15998, 36
p = 16001

G = (4971 , 2914 , 1)
q = 16097

In [12]:
kpub, kpriv = keysECDSA(p, a, b, G, q)
m = 25

print("Clave publica:", kpub)
print("Clave privada:", kpriv)

Clave publica: (16001, 15998, 36, 16097, (4971, 2914, 1), (5119, 5678, 1))
Clave privada: 4274


In [13]:
firma = firmaECDSA(m, kpriv, kpub)

print("Firma:", firma)

Firma: (10676, 3343)


In [14]:
print("Verificacion:", verificarECDSA(kpub, m, firma))

Verificacion: True
